<a href="https://colab.research.google.com/github/UAMCAntwerpen/2040FBDBIC/blob/main/03_Chemical_informatics_with_RDKit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installation of RDKit in Google Colab

In [ ]:
import sys
import os
import requests
import subprocess
import shutil
from logging import getLogger, StreamHandler, INFO

logger = getLogger(__name__)
logger.addHandler(StreamHandler())
logger.setLevel(INFO)

def install(
        chunk_size=4096,
        file_name="Miniconda3-latest-Linux-x86_64.sh",
        url_base="https://repo.continuum.io/miniconda/",
        conda_path=os.path.expanduser(os.path.join("~", "miniconda")),
        rdkit_version=None,
        add_python_path=True,
        force=False):

    python_path = os.path.join(
        conda_path,
        "lib",
        "python{0}.{1}".format(*sys.version_info),
        "site-packages",
    )

    if add_python_path and python_path not in sys.path:
        logger.info("add {} to PYTHONPATH".format(python_path))
        sys.path.append(python_path)

    if os.path.isdir(os.path.join(python_path, "rdkit")):
        logger.info("rdkit is already installed")
        if not force:
            return

        logger.info("force re-install")

    url = url_base + file_name
    python_version = "{0}.{1}.{2}".format(*sys.version_info)

    logger.info("python version: {}".format(python_version))

    if os.path.isdir(conda_path):
        logger.warning("remove current miniconda")
        shutil.rmtree(conda_path)
    elif os.path.isfile(conda_path):
        logger.warning("remove {}".format(conda_path))
        os.remove(conda_path)

    logger.info('fetching installer from {}'.format(url))
    res = requests.get(url, stream=True)
    res.raise_for_status()
    with open(file_name, 'wb') as f:
        for chunk in res.iter_content(chunk_size):
            f.write(chunk)
    logger.info('done')

    logger.info('installing miniconda to {}'.format(conda_path))
    subprocess.check_call(["bash", file_name, "-b", "-p", conda_path])
    logger.info('done')

    logger.info("installing rdkit")
    subprocess.check_call([
        os.path.join(conda_path, "bin", "conda"),
        "install",
        "--yes",
        "-c", "rdkit",
        "python=={}".format(python_version),
        "rdkit" if rdkit_version is None else "rdkit=={}".format(rdkit_version)])
    logger.info("done")

    import rdkit
    logger.info("rdkit-{} installation finished!".format(rdkit.__version__))


install()

add /root/miniconda/lib/python3.7/site-packages to PYTHONPATH
python version: 3.7.10
fetching installer from https://repo.continuum.io/miniconda/Miniconda3-latest-Linux-x86_64.sh
done
installing miniconda to /root/miniconda
done
installing rdkit
done
rdkit-2020.09.1 installation finished!


In [ ]:
# Import statements
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole

## Your first RDKit lines

In [ ]:
from rdkit import Chem

In [ ]:
mol = Chem.MolFromSmiles("CCCC")
print(mol)
mol

In [ ]:
print("number of atoms:", mol.GetNumAtoms())
nBonds = mol.GetNumBonds()
print("number of bonds:", nBonds)

In [ ]:
print(Chem.MolToSmiles(mol))

In [ ]:
for smiles in ["CCCC", "c"]:
  print("Smiles:", smiles)
  mol = Chem.MolFromSmiles(smiles)
  print(mol)
  print(mol is None)
  print("")

In [ ]:
from rdkit.Chem import AllChem
mol = Chem.MolFromSmiles("C")
print("number of atoms:", mol.GetNumAtoms())
print("number of bonds:", mol.GetNumBonds())
print("")
mol = AllChem.AddHs(mol)
print("number of atoms:", mol.GetNumAtoms())
print("number of bonds:", mol.GetNumBonds())
print("")
mol = AllChem.RemoveHs(mol)
print("number of atoms:", mol.GetNumAtoms())
print("number of bonds:", mol.GetNumBonds())

Setting properties:

In [ ]:
mol = Chem.MolFromSmiles("c1ccccc1")
mol.SetProp("_Name", "benzene")
print(Chem.MolToMolBlock(mol))

## Looping over atoms and bonds

In [ ]:
mol = Chem.MolFromSmiles('C1OC1')
for atom in mol.GetAtoms():
  print(atom.GetAtomicNum(), atom.GetIdx(), atom.GetSymbol(), atom.GetExplicitValence())

Atom indices:

In [ ]:
for i in range(0, mol.GetNumAtoms()):
  print(i, mol.GetAtomWithIdx(i).GetSymbol())

Atom neighbors:

In [ ]:
for atom in mol.GetAtoms():
  neighbors = atom.GetNeighbors()
  print(neighbors)
  print(atom.GetIdx(), end = ": ")
  for neighbor in neighbors: print(neighbor.GetIdx(), end="-")
  print("")

Bonds:

In [ ]:
for bond in mol.GetBonds():
  bt = bond.GetBondType()
  bbi = bond.GetBeginAtomIdx()
  bei = bond.GetEndAtomIdx()
  print(bt, bbi, "-", bei)

## Rings

In [ ]:
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.drawOptions.addAtomIndices = True
mol = Chem.MolFromSmiles('OC1C2C1CC2')
mol

In [ ]:
for atom in mol.GetAtoms():
  idx = atom.GetIdx()
  ring = atom.IsInRing()
  r3 = atom.IsInRingSize(3)
  r4 = atom.IsInRingSize(4)
  r6 = atom.IsInRingSize(6)
  print(idx, ring, r3, r4, r6)

In [ ]:
mol = Chem.MolFromSmiles("OC1C2C1CC2")
smallestSetOfSmallestRings = Chem.GetSymmSSSR(mol)
n_sssr = len(smallestSetOfSmallestRings)
print(n_sssr)
for i in range(n_sssr): print(list(smallestSetOfSmallestRings[i]))
mol

In [ ]:
mol = Chem.MolFromSmiles("c12ccccc1cccc2")
smallestSetOfSmallestRings = Chem.GetSymmSSSR(mol)
n_sssr = len(smallestSetOfSmallestRings)
print(n_sssr)
for i in range(n_sssr): print(list(smallestSetOfSmallestRings[i]))
mol

In [ ]:
mol = Chem.MolFromSmiles("[C@H]12[C@@H]3[C@@H]4[C@H]1[C@H]5[C@@H]4[C@H]3[C@@H]25")
smallestSetOfSmallestRings = Chem.GetSymmSSSR(mol)
n_sssr = len(smallestSetOfSmallestRings)
print(n_sssr)
for i in range(n_sssr): print(list(smallestSetOfSmallestRings[i]))
mol

## Reading and writing molecules

Single molecules:

In [ ]:
mol = Chem.MolFromMolFile(“input.sdf”)

In [ ]:
mol = Chem.MolFromSmiles("c1ccccc1")
mol is None

In [ ]:
mol = Chem.MolFromSmiles("c1cCc1")
mol is None

In [ ]:
smiles = ['c1ccccc1', 'c1cCc1']
mols = []
for s in smiles:
  mol = Chem.MolFromSmiles(s)
  if mol is None:
    continue
  else:
    mols.append(mol)
print(len(mols))

InChi strings:

In [ ]:
from rdkit.Chem import inchi
mol = Chem.MolFromSmiles("C1CCNCC1")
inchistring = inchi.MolToInchi(mol)
print(inchistring)
mol = inchi.MolFromInchi(inchistring)
print(Chem.MolToSmiles(mol))

MOL blocks:

In [ ]:
mol = Chem.MolFromSmiles('C1CCC1')
print(Chem.MolToMolBlock(mol))

Property data:

In [ ]:
mol.SetProp("_Name","cyclobutane")
print(Chem.MolToMolBlock(mol))

In [ ]:
mol.GetProp("_Name")

Sets of molecules:

In [ ]:
smiles = ["C", "CC", "CO", "CCO", "C1OC1"]
mols = []
for s in smiles: mols.append(Chem.MolFromSmiles(s))
writer = Chem.SDWriter("filename.sdf")
for mol in mols: writer.write(mol)
writer.close()

In [ ]:
!cat filename.sdf

In [ ]:
reader = Chem.SDMolSupplier("filename.sdf")
for mol in reader:
  if mol is None: continue
  print(Chem.MolToSmiles(mol))

In [ ]:
f = open("file.smi", "w")
for mol in mols: f.write(Chem.MolToSmiles(mol) + "\n")
f.close()

In [ ]:
!cat file.smi

## Working with conformations

In [ ]:
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
mol.SetProp("_Name", "aspirine")
print(Chem.MolToMolBlock(mol))

In [ ]:
mol = Chem.AddHs(mol)
print(Chem.MolToMolBlock(mol))

In [ ]:
AllChem.EmbedMolecule(mol)
print(Chem.MolToMolBlock(mol))

In [ ]:
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
mol = Chem.AddHs(mol)
conformationIds = AllChem.EmbedMultipleConfs(mol, numConfs=10)
print(len(conformationIds))
w = Chem.SDWriter("aspirin.sdf")
for cid in conformationIds: w.write(mol, confId = cid)
w.close()

In [ ]:
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
mol = Chem.AddHs(mol)
conformationIds = AllChem.EmbedMultipleConfs(mol, numConfs=10)
rmslist = []
AllChem.AlignMolConformers(mol, RMSlist = rmslist)
for rms in rmslist: print(rms)
w = Chem.SDWriter("aspirin.sdf")
for cid in conformationIds: w.write(mol, confId = cid)
w.close()

## Substructure searching: SMARTS

In [ ]:
m = Chem.MolFromSmiles('c1ccccc1O')
m

In [ ]:
smartsMol = Chem.MolFromSmarts('ccO')
m.HasSubstructMatch(smartsMol)

In [ ]:
m.GetSubstructMatch(smartsMol)

In [ ]:
m.GetSubstructMatches(smartsMol)

In [ ]:
bromophenols = ["Oc1ccccc1Br", "Oc1cccc(Br)c1", "Oc1ccc(Br)cc1"]
mols = []
for bp in bromophenols: mols.append(Chem.MolFromSmiles(bp))

In [ ]:
p = Chem.MolFromSmarts("Br[$(c1c([OH])cccc1),$(c1ccc([OH])cc1)]")
for mol in mols:
  if mol.HasSubstructMatch(p):
    print(Chem.MolToSmiles(mol), "True")
  else:
    print(Chem.MolToSmiles(mol), "False")